# 01 Baseline Inference: Zero-Shot vs Few-Shot

In [1]:
# setup and imports
import sys
import random
import importlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import torch
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="transformers")

# Keep HF logs clean while we focus on model outputs.
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import src.model as model_module
import src.evaluate as eval_module
importlib.reload(model_module)
importlib.reload(eval_module)

from src.model import DomainSummarizer, GenerationParams, FewShotExample
from src.evaluate import compute_rouge_batch, format_comparison_row, to_markdown_table

RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_FILE = RESULTS_DIR / "comparison_table.md"

print(f"Device available: {'cuda' if torch.cuda.is_available() else 'cpu'}")

Device available: cuda


In [2]:
# load a fixed 70-sample test subset and few-shot demonstrations
NUM_SAMPLES = 20

print("Loading test split...")
test_ds = load_dataset("cnn_dailymail", "3.0.0", split="test")
test_subset = test_ds.shuffle(seed=SEED).select(range(NUM_SAMPLES))

print("Loading tiny validation slice for few-shot demonstrations...")
val_demo = load_dataset("cnn_dailymail", "3.0.0", split="validation[:2]")
few_shot_examples = [
    FewShotExample(article=item["article"], summary=item["highlights"])
    for item in val_demo
]

articles = [item["article"] for item in test_subset]
references = [item["highlights"] for item in test_subset]

print(f"Prepared {len(articles)} test samples and {len(few_shot_examples)} few-shot examples.")

Loading test split...


Loading tiny validation slice for few-shot demonstrations...
Prepared 20 test samples and 2 few-shot examples.


In [3]:
# initialize summarizer and generation config
BASE_MODEL_NAME = "google/flan-t5-base"
# Deterministic baseline settings for fair ROUGE comparison.
generation_cfg = GenerationParams(
    temperature=0.0,
    max_new_tokens=128,
)

summarizer = DomainSummarizer(model_name=BASE_MODEL_NAME, seed=SEED)

try:
    summarizer.load()
    print(f"Loaded model: {BASE_MODEL_NAME}")
except Exception as exc:
    raise RuntimeError(
        "Model loading failed. Check internet connectivity, available memory, "
        "and Hugging Face access."
    ) from exc

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

Loaded model: google/flan-t5-base


In [4]:
# few-shot prompt length sanity check (pre-run)
sample_prompt = summarizer.build_few_shot_prompt(articles[0], few_shot_examples)
sample_tokens = summarizer._token_len(sample_prompt)
print(f"Sample few-shot prompt tokens: {sample_tokens}")
print(f"Configured max input tokens: {summarizer.max_input_tokens}")
if sample_tokens > summarizer.max_input_tokens:
    print("Prompt is longer than max input tokens; truncation will be applied safely during summarize().")

Sample few-shot prompt tokens: 480
Configured max input tokens: 512


In [5]:
# zero-shot summarization
zero_shot_predictions = []
zero_shot_errors = 0

for article in tqdm(articles, desc="Zero-shot"):
    try:
        pred = summarizer.summarize(article=article, generation=generation_cfg)
    except Exception as exc:
        pred = ""
        zero_shot_errors += 1
        print(f"Zero-shot generation error: {exc}")
    zero_shot_predictions.append(pred)

print(f"Zero-shot complete: {len(zero_shot_predictions)} predictions, {zero_shot_errors} errors")

Zero-shot:   0%|          | 0/20 [00:00<?, ?it/s]

Zero-shot complete: 20 predictions, 0 errors


In [6]:
# few-shot summarization
few_shot_predictions = []
few_shot_errors = 0

for article in tqdm(articles, desc="Few-shot"):
    try:
      
        pred = summarizer.summarize(
            article=article,
            generation=generation_cfg,
            few_shot_examples=few_shot_examples,
        )
    except Exception as exc:
        pred = ""
        few_shot_errors += 1
        print(f"Few-shot generation error: {exc}")
    few_shot_predictions.append(pred)

print(f"Few-shot complete: {len(few_shot_predictions)} predictions, {few_shot_errors} errors")

Few-shot:   0%|          | 0/20 [00:00<?, ?it/s]

Few-shot complete: 20 predictions, 0 errors


In [7]:
# quick quality guard: detect collapse if many few-shot outputs are identical
unique_few_shot = len({p.strip() for p in few_shot_predictions if p.strip()})
collapse_ratio = unique_few_shot / max(1, len(few_shot_predictions))
print(f"Unique few-shot outputs: {unique_few_shot}/{len(few_shot_predictions)} ({collapse_ratio:.2%})")
if collapse_ratio < 0.5:
    print("WARNING: Few-shot collapse detected. Review prompts/examples or generation settings.")

Unique few-shot outputs: 20/20 (100.00%)


In [8]:
# compute ROUGE metrics and prepare comparison table
zero_shot_metrics = compute_rouge_batch(references, zero_shot_predictions)
few_shot_metrics = compute_rouge_batch(references, few_shot_predictions)

rows = [
    format_comparison_row(
        model_label="FLAN-T5 Base (Zero-shot)",
        metrics=zero_shot_metrics,
        sample_count=NUM_SAMPLES,
        notes=f"temp={generation_cfg.temperature}, top_k={generation_cfg.top_k}, top_p={generation_cfg.top_p}, errors={zero_shot_errors}",
    ),
    format_comparison_row(
        model_label="FLAN-T5 Base (Few-shot)",
        metrics=few_shot_metrics,
        sample_count=NUM_SAMPLES,
        notes=f"2 demos, temp={generation_cfg.temperature}, errors={few_shot_errors}",
    ),
]

comparison_table_md = to_markdown_table(rows)
comparison_df = pd.DataFrame(rows)
comparison_df

,model,rouge1,rouge2,rougeL,samples,notes
0,FLAN-T5 Base (Zero-shot),0.3300,0.1316,0.2324,20,"temp=0.0, top_k=None, top_p=None, errors=0"
1,FLAN-T5 Base (Few-shot),0.3300,0.1316,0.2324,20,"2 demos, temp=0.0, errors=0"


In [9]:
# save baseline results to markdown
run_time = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
content = "\n".join([
    "# Baseline vs Fine-Tuned ROUGE Comparison",
    "",
    "## Baseline (Before Fine-Tuning)",
    "",
    f"Generated at: {run_time}",
    "",
    comparison_table_md,
    "",
    "## Notes",
    f"- Dataset: cnn_dailymail test subset ({NUM_SAMPLES} samples)",
    "- Base model: google/flan-t5-base",
    "- Fine-tuned section will be appended by notebook 03",
])

COMPARISON_FILE.write_text(content, encoding="utf-8")
print(f"Saved baseline comparison table to: {COMPARISON_FILE}")

Saved baseline comparison table to: d:\GenAI_DeepLearning\domain-summarizer\results\comparison_table.md


In [10]:
# inspect qualitative examples
preview_rows = []
for i in range(min(3, NUM_SAMPLES)):
    preview_rows.append({
        "sample_idx": i,
        "reference": references[i][:300],
        "zero_shot": zero_shot_predictions[i][:300],
        "few_shot": few_shot_predictions[i][:300],
    })

pd.DataFrame(preview_rows)

,sample_idx,reference,zero_shot,few_shot
0,0,CNN's Dr. Sanjay Gupta says we should legalize...,Marijuana is a medical marijuana that has been...,Marijuana is a medical marijuana that has been...
1,1,Child has amassed thousands of Twitter followe...,"Little boy from Memphis, Tennessee, poses with...","Little boy from Memphis, Tennessee, poses with..."
2,2,The presidential hopeful held a town hall meet...,New Jersey Gov Chris Christie is being called ...,New Jersey Gov Chris Christie is being called ...


In [11]:
# cleanup
summarizer.unload()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Cleanup complete.")

Cleanup complete.
